# HMM Activity Recognition

I use accelerometer and gyroscope data to train a Hidden Markov Model that detects standing, walking, jumping, and still activities.

In [ ]:
# I import the libraries I need
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from scipy.optimize import linear_sum_assignment
from hmmlearn.hmm import GaussianHMM
from sklearn.metrics import confusion_matrix, classification_report

# I set up folders and activity names
DATA_DIR = "data"
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

ACTIVITIES = ["standing", "walking", "jumping", "still"]

## 1. Load and explore data

In [ ]:
def load_recording(activity, number):
    """I load one recording by merging accel and gyro CSV files."""
    num_str = f"{number:02d}"
    accel_path = os.path.join(DATA_DIR, activity, f"{activity}_{num_str}_accel.csv")
    gyro_path = os.path.join(DATA_DIR, activity, f"{activity}_{num_str}_gyro.csv")

    if not os.path.exists(accel_path) or not os.path.exists(gyro_path):
        return None

    accel = pd.read_csv(accel_path)
    gyro = pd.read_csv(gyro_path)

    # I round time so accel and gyro rows line up
    accel["seconds_elapsed"] = accel["seconds_elapsed"].round(2)
    gyro["seconds_elapsed"] = gyro["seconds_elapsed"].round(2)

    merged = pd.merge(accel, gyro, on="seconds_elapsed", suffixes=("_accel", "_gyro"))
    merged = merged.rename(columns={
        "x_accel": "accel_x", "y_accel": "accel_y", "z_accel": "accel_z",
        "x_gyro": "gyro_x", "y_gyro": "gyro_y", "z_gyro": "gyro_z"
    })
    return merged

In [ ]:
# I load one sample file and check the sampling rate
sample = load_recording("walking", 1)
dt = sample["seconds_elapsed"].diff().mean()
sample_rate = 1 / dt
print(f"My sampling rate is about {sample_rate:.1f} Hz")
print(f"This recording has {len(sample)} rows and lasts {sample['seconds_elapsed'].max():.1f} seconds")
sample.head()

In [ ]:
# I plot raw accel and gyro magnitude for one recording per activity
fig, axes = plt.subplots(4, 2, figsize=(12, 14))

for i, activity in enumerate(ACTIVITIES):
    df = load_recording(activity, 1)
    accel_mag = np.sqrt(df["accel_x"]**2 + df["accel_y"]**2 + df["accel_z"]**2)
    gyro_mag = np.sqrt(df["gyro_x"]**2 + df["gyro_y"]**2 + df["gyro_z"]**2)

    axes[i, 0].plot(df["seconds_elapsed"], accel_mag)
    axes[i, 0].set_title(f"{activity} - accelerometer")
    axes[i, 0].set_xlabel("seconds")
    axes[i, 0].set_ylabel("magnitude")

    axes[i, 1].plot(df["seconds_elapsed"], gyro_mag)
    axes[i, 1].set_title(f"{activity} - gyroscope")
    axes[i, 1].set_xlabel("seconds")
    axes[i, 1].set_ylabel("magnitude")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "raw_sensor_samples.png"), dpi=150)
plt.show()

## 2. Windowing

I use 1-second windows with 0.5-second steps. At ~100 Hz that means 100 samples per window.

In [ ]:
# I set window size based on my sampling rate
WINDOW_SEC = 1.0
STEP_SEC = 0.5
window_samples = int(sample_rate * WINDOW_SEC)
step_samples = int(sample_rate * STEP_SEC)

print(f"Window size: {window_samples} samples ({WINDOW_SEC} s)")
print(f"Step size: {step_samples} samples ({STEP_SEC} s)")

In [ ]:
def make_windows(df):
    """I cut a recording into overlapping windows."""
    windows = []
    n = len(df)
    start = 0
    while start + window_samples <= n:
        chunk = df.iloc[start:start + window_samples]
        windows.append(chunk)
        start += step_samples
    return windows

## 3. Feature extraction

I compute time-domain and frequency-domain features from both accelerometer and gyroscope data in each window.

In [ ]:
def extract_features(window_df, rate):
    """I compute 10 features from accel and gyro data in one window."""
    ax = window_df["accel_x"].values
    ay = window_df["accel_y"].values
    az = window_df["accel_z"].values
    accel_mag = np.sqrt(ax**2 + ay**2 + az**2)

    gx = window_df["gyro_x"].values
    gy = window_df["gyro_y"].values
    gz = window_df["gyro_z"].values
    gyro_mag = np.sqrt(gx**2 + gy**2 + gz**2)

    # Time-domain features (accelerometer)
    mean_mag = np.mean(accel_mag)
    std_mag = np.std(accel_mag)
    var_mag = np.var(accel_mag)
    sma = np.mean(np.abs(ax) + np.abs(ay) + np.abs(az))

    if np.std(ax) > 0 and np.std(ay) > 0:
        corr_xy = pearsonr(ax, ay)[0]
    else:
        corr_xy = 0.0

    # Time-domain features (gyroscope)
    gyro_mean = np.mean(gyro_mag)
    gyro_var = np.var(gyro_mag)

    # Frequency-domain features (FFT on accel magnitude)
    fft_vals = np.abs(np.fft.rfft(accel_mag))
    freqs = np.fft.rfftfreq(len(accel_mag), d=1 / rate)
    dom_freq = freqs[1:][np.argmax(fft_vals[1:])]
    spec_energy = np.sum(fft_vals**2)
    fft_peak = np.max(fft_vals[1:])

    return [
        mean_mag, std_mag, var_mag, sma, corr_xy,
        gyro_mean, gyro_var,
        dom_freq, spec_energy, fft_peak
    ]

FEATURE_NAMES = [
    "mean_mag", "std_mag", "var_mag", "sma", "corr_xy",
    "gyro_mean", "gyro_var",
    "dom_freq", "spec_energy", "fft_peak"
]

In [ ]:
def get_recording_numbers(activity, split):
    """I return recording numbers for train or test split."""
    if split == "train":
        nums = list(range(1, 9))
        if activity == "walking":
            nums.remove(7)  # I skip missing accel file
        return nums
    return [9, 10]


def build_dataset(split):
    """I build feature rows for all recordings in a split."""
    rows = []
    for act_idx, activity in enumerate(ACTIVITIES):
        for num in get_recording_numbers(activity, split):
            df = load_recording(activity, num)
            if df is None:
                continue
            for w in make_windows(df):
                feat = extract_features(w, sample_rate)
                rows.append({
                    "activity": activity,
                    "activity_id": act_idx,
                    "recording": num,
                    "features": feat
                })
    return rows

In [ ]:
# I build train and test sets
train_rows = build_dataset("train")
test_rows = build_dataset("test")

print(f"Train windows: {len(train_rows)}")
print(f"Test windows: {len(test_rows)}")

## 4. Normalize features (z-score)

I fit normalization on training data only, then apply it to train and test.

In [ ]:
# I stack training features into a matrix
X_train_raw = np.array([r["features"] for r in train_rows])
X_test_raw = np.array([r["features"] for r in test_rows])

# I compute mean and std from training data only
train_mean = X_train_raw.mean(axis=0)
train_std = X_train_raw.std(axis=0)
train_std[train_std == 0] = 1.0

# I apply z-score normalization
X_train = (X_train_raw - train_mean) / train_std
X_test = (X_test_raw - train_mean) / train_std

y_train = np.array([r["activity_id"] for r in train_rows])
y_test = np.array([r["activity_id"] for r in test_rows])

## 5. Train HMM (Baum–Welch)

I estimate emission parameters (B) from labeled training windows. Baum–Welch then learns the transition matrix (A) and initial probabilities (pi).

In [ ]:
# I group training windows by recording for sequence lengths
lengths = []
start = 0
prev_key = None
for i, row in enumerate(train_rows):
    key = (row["activity"], row["recording"])
    if prev_key is not None and key != prev_key:
        lengths.append(i - start)
        start = i
    prev_key = key
lengths.append(len(train_rows) - start)

print(f"I have {len(lengths)} training sequences")
print(f"Total training windows: {sum(lengths)}")

In [ ]:
# I estimate emission means and variances from labeled training windows
emission_means = np.zeros((4, X_train.shape[1]))
emission_covars = np.zeros((4, X_train.shape[1]))
for act_id in range(4):
    mask = y_train == act_id
    emission_means[act_id] = X_train[mask].mean(axis=0)
    emission_covars[act_id] = X_train[mask].var(axis=0) + 1e-6

# I train transitions (A) and pi with Baum-Welch; emissions (B) stay fixed
model = GaussianHMM(
    n_components=4,
    covariance_type="diag",
    n_iter=100,
    tol=1e-4,
    random_state=42,
    init_params="",
    params="st"
)
model.means_ = emission_means
model._init(X_train, lengths)
model._covars_ = emission_covars
model.fit(X_train, lengths=lengths)

print(f"Baum-Welch converged: {model.monitor_.converged}")
print(f"Iterations used: {model.monitor_.iter}")
print(f"Final log-likelihood: {model.score(X_train, lengths=lengths):.2f}")
print(f"Initial state probabilities (pi): {model.startprob_}")

In [ ]:
# I plot log-likelihood history from Baum-Welch
if len(model.monitor_.history) > 0:
    plt.figure(figsize=(8, 4))
    plt.plot(model.monitor_.history)
    plt.title("Baum-Welch convergence (log-likelihood)")
    plt.xlabel("iteration")
    plt.ylabel("log-likelihood")
    plt.savefig(os.path.join(FIG_DIR, "baum_welch_convergence.png"), dpi=150)
    plt.show()

# I plot initial state probabilities (pi)
pi_labels = [f"state {i}" for i in range(4)]
plt.figure(figsize=(8, 4))
plt.bar(pi_labels, model.startprob_)
plt.title("Initial state probabilities (pi)")
plt.xlabel("HMM state")
plt.ylabel("probability")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "initial_state_probs.png"), dpi=150)
plt.show()

## 6. Map HMM states to activities and decode (Viterbi)

HMM states are numbered 0-3. I map each state to the activity it matches best in training data. Then I use Viterbi (`predict`) on test data.

In [ ]:
# I decode training data with Viterbi to learn state labels
train_states = model.predict(X_train, lengths=lengths)

# I build a score matrix: how often each state matches each activity
score_matrix = np.zeros((4, 4))
for state in range(4):
    mask = train_states == state
    if mask.sum() > 0:
        for act_id in range(4):
            score_matrix[state, act_id] = np.sum(y_train[mask] == act_id)

# I pick the best one-to-one state-to-activity map
row_idx, col_idx = linear_sum_assignment(-score_matrix)
state_to_activity = {int(row_idx[i]): int(col_idx[i]) for i in range(4)}
activity_to_state = {v: k for k, v in state_to_activity.items()}

print("State to activity map:", state_to_activity)
for i, act in enumerate(ACTIVITIES):
    print(f"  {act} -> state {activity_to_state.get(i, '?')}")

In [ ]:
# I build test sequence lengths
test_lengths = []
start = 0
prev_key = None
for i, row in enumerate(test_rows):
    key = (row["activity"], row["recording"])
    if prev_key is not None and key != prev_key:
        test_lengths.append(i - start)
        start = i
    prev_key = key
test_lengths.append(len(test_rows) - start)

# I run Viterbi on unseen test data
test_states = model.predict(X_test, lengths=test_lengths)
y_pred = np.array([state_to_activity[s] for s in test_states])

## 7. Visualizations

In [ ]:
# I plot the transition matrix as a heatmap
labelled_trans = np.zeros((4, 4))
for i in range(4):
    for j in range(4):
        si = activity_to_state.get(i, i)
        sj = activity_to_state.get(j, j)
        labelled_trans[i, j] = model.transmat_[si, sj]

plt.figure(figsize=(8, 6))
sns.heatmap(labelled_trans, annot=True, fmt=".2f", xticklabels=ACTIVITIES, yticklabels=ACTIVITIES, cmap="Blues")
plt.title("Transition matrix (A)")
plt.xlabel("to activity")
plt.ylabel("from activity")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "transition_matrix.png"), dpi=150)
plt.show()

In [ ]:
# I plot emission means (feature values per activity state)
emission_means = np.zeros((4, len(FEATURE_NAMES)))
for act_id in range(4):
    state = activity_to_state.get(act_id, act_id)
    emission_means[act_id] = model.means_[state]

plt.figure(figsize=(14, 5))
sns.heatmap(emission_means, annot=True, fmt=".2f", xticklabels=FEATURE_NAMES, yticklabels=ACTIVITIES, cmap="YlOrRd")
plt.title("Emission means per activity (normalized features)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "emission_means.png"), dpi=150)
plt.show()

In [ ]:
# I plot decoded activity sequence for one unseen test recording
example = load_recording("walking", 9)
example_windows = make_windows(example)
example_feats = np.array([extract_features(w, sample_rate) for w in example_windows])
example_feats = (example_feats - train_mean) / train_std
example_states = model.predict(example_feats)
example_pred = [ACTIVITIES[state_to_activity[s]] for s in example_states]

time_axis = np.arange(len(example_pred)) * STEP_SEC
plt.figure(figsize=(10, 3))
plt.step(time_axis, example_pred, where="post")
plt.title("Viterbi decoded sequence - unseen walking_09")
plt.xlabel("time (s)")
plt.ylabel("predicted activity")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "decoded_sequence.png"), dpi=150)
plt.show()

In [ ]:
# I plot confusion matrix on unseen test windows
cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2, 3])
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=ACTIVITIES, yticklabels=ACTIVITIES, cmap="Greens")
plt.title("Confusion matrix (unseen test data)")
plt.xlabel("predicted")
plt.ylabel("true")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

## 8. Evaluation metrics on unseen data

Test recordings 09 and 10 were not used in training.

In [ ]:
print(classification_report(y_test, y_pred, target_names=ACTIVITIES))

In [ ]:
# I compute sensitivity, specificity, and per-activity accuracy
results = []
for act_id, activity in enumerate(ACTIVITIES):
    tp = np.sum((y_test == act_id) & (y_pred == act_id))
    fn = np.sum((y_test == act_id) & (y_pred != act_id))
    tn = np.sum((y_test != act_id) & (y_pred != act_id))
    fp = np.sum((y_test != act_id) & (y_pred == act_id))
    n_samples = np.sum(y_test == act_id)

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    class_accuracy = tp / n_samples if n_samples > 0 else 0.0

    results.append({
        "Activity": activity,
        "Samples": n_samples,
        "Sensitivity": round(sensitivity, 3),
        "Specificity": round(specificity, 3),
        "Overall Accuracy": round(class_accuracy, 3)
    })

metrics_df = pd.DataFrame(results)
overall_accuracy = np.mean(y_test == y_pred)
print(metrics_df.to_string(index=False))
print(f"\nOverall test accuracy (all windows): {overall_accuracy:.3f}")
metrics_df.to_csv(os.path.join(FIG_DIR, "evaluation_metrics.csv"), index=False)